# 초구 예측 — model_17 (3단계: 위치 상황 보정을 단계적 분류로)

- 지금 (model_16): model_13 위치 24칸 확률을 출발점으로 24칸 LightGBM 하나가 상황 보정, 월 단위 재학습
- 0장 0-1 (시즌 단위, 2019): **(나) H/S/C 3분류 × 그룹 안 LightGBM 3개**가 24칸 한 번에보다 +2.12 ×1000 (95% +0.86 ~ +3.41), H/S/C 부분·그룹 안 부분 모두 좋아짐
- 모델: P(칸) = P(H/S/C) × P(칸 | H/S/C) (사용자 결정)
  - H/S/C 3분류: 출발점 = log model_13 H/S/C 확률
  - 그룹 안 3개 (H 4칸 · S 12칸 · C 8칸): 그 그룹 공만으로 학습, 출발점 = log model_13 그룹 안 비율
  - 변수는 model_14과 같음 (상황 9개 + 구종 원-핫·손)
  - 네 모델 각각 LightGBM 파라미터·트리 수를 2019로 고름 (학습 2018, 연결된 파라미터 함께)
  - 월 단위 재학습 (model_16과 같은 방식)
- 위치 정규분포(좌표)·구종은 model_16 그대로 -> 24칸 확률만 바뀜
- 비교 대상: model_16

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_17"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. 위치 단계적 분류 (H/S/C -> 24칸)

- 24칸 log loss = **H/S/C 부분** (−log P(실제 H/S/C)) + **그룹 안 부분** (−log P(실제 칸 | 실제 H/S/C))
- 비교 (모두 실제 구종 기준, 변수는 model_14 상황 변수 + 구종·손, 파라미터 잎 7 · 잎당 최소 20 · λ 1000, 학습 2018, 트리 수는 2019 조기 종료 = model_14과 같은 조건)
  - model_13: 보정 없음
  - **24칸 한 번에** = model_14 구조 (트리 168, 재현 확인)
  - **(가) H/S/C 3분류 LightGBM × model_13 그룹 안 비율**: 출발점 = log model_13 H/S/C 확률, 그룹 안은 그대로
  - **(나) (가) × 그룹 안 LightGBM 3개**: H(4칸)·S(12칸)·C(8칸)마다 그 그룹 공만으로 칸을 학습, 출발점 = log model_13 그룹 안 비율
- 이득 ×1000 = 24칸 한 번에 − 후보 (양수 = 후보가 좋음), 경기 단위 95% 구간

In [2]:
DIAG = "model_16"
pp18 = fc.load_results(["model_14"])["model_14"]  # 24칸 한 번에 (시즌 단위) 재현 확인용
LOC_REF, LOC_REF_HL = "decay", 173.0  # model_13 위치 정규분포
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0
LOC_SEASON_F = 0.707
LOC_AC = (None, 12.5, 17.3)
LOC_VAR = (200.0, 9.74, False, 316.0)
ZONE_M = 32
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]
LOC_FEAT18 = CLASSES + ["same_side"] + LOC_SIT
LOC_LGB18, LOC_ROUNDS18 = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}, 168
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g



LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)
for yr in (2018, fc.VAL_YEAR):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
from scipy.special import log_softmax
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]]], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)
XLOC = XLOC[LOC_FEAT18]
yr_l = fp["GYEAR"].to_numpy()
itr, i19 = np.flatnonzero(yr_l == 2018), np.flatnonzero(val)
gid_all = fp["gameID"].to_numpy()
GIDX = pd.Series(fc.ZONE_GROUP).map({"H": 0, "S": 1, "C": 2}).to_numpy()  # 칸 -> 그룹 번호
g_all = GIDX[y_zone]  # 공마다 실제 그룹
BASE = dict(learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0, deterministic=True,
            force_row_wise=True, num_threads=4, **LOC_LGB18)

def center(L):  # 공마다 상수 빼기 (softmax에 영향 없음)
    return L - L.mean(1, keepdims=True)

def fit_mc(tr, va, y, init, ncls, rounds=None):  # 다분류 LightGBM: rounds 없으면 va로 조기 종료
    p = {**BASE, "objective": "multiclass", "num_class": ncls}
    dtr = lgb.Dataset(XLOC.iloc[tr], y[tr], init_score=init[tr], free_raw_data=False)
    if rounds is None:
        dva = lgb.Dataset(XLOC.iloc[va], y[va], init_score=init[va], reference=dtr)
        return lgb.train(p, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train(p, dtr, rounds)

def pred_mc(bst, idx, init, n=None):
    return np.exp(log_softmax(bst.predict(XLOC.iloc[idx], raw_score=True, num_iteration=n or bst.best_iteration or None) + init[idx], axis=1))

# 24칸 한 번에 (model_14 구조)
INIT24 = center(np.log(np.clip(PZ_TRUE, 1e-6, None)))
b24 = fit_mc(itr, None, y_zone, INIT24, len(ZONES), LOC_ROUNDS18)
P24 = pred_mc(b24, i19, INIT24, LOC_ROUNDS18)
i9 = np.arange(len(i19))
assert np.allclose(-np.log(np.clip(P24[i9, y_zone[i19]], 1e-12, None)), fc.align(pp18, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy(), atol=1e-6), "model_14 재현 실패"
print("24칸 한 번에 = model_14 (재현 확인)")

# (가) H/S/C 3분류
PG17 = PZ_TRUE @ fc.GROUP2  # model_13 H/S/C 확률 (n, 3)
INIT3 = center(np.log(np.clip(PG17, 1e-6, None)))
b3 = fit_mc(itr, i19, g_all, INIT3, 3)
PG_A = pred_mc(b3, i19, INIT3)
W17 = PZ_TRUE / np.clip(PG17[:, GIDX], 1e-12, None)  # model_13 그룹 안 비율 P(칸 | 그룹) (n, 24)
P_A = PG_A[:, GIDX] * W17[i19]  # 3분류 × 그룹 안 그대로

# (나) 그룹 안 LightGBM 3개
W_B = W17[i19].copy()
nt_b = {}
for gk, gname in enumerate(fc.HSC):
    zk = np.flatnonzero(GIDX == gk)  # 그 그룹의 칸 번호
    loc_y = np.full(len(fp), -1)
    loc_y[np.isin(y_zone, zk)] = pd.Series(np.arange(len(zk)), index=zk).reindex(y_zone[np.isin(y_zone, zk)]).to_numpy()  # 그룹 안 칸 번호
    init_g = center(np.log(np.clip(W17[:, zk], 1e-6, None)))
    tr_g, va_g = itr[g_all[itr] == gk], i19[g_all[i19] == gk]
    bg = fit_mc(tr_g, va_g, loc_y, init_g, len(zk))
    nt_b[gname] = bg.best_iteration
    W_B[:, zk] = pred_mc(bg, i19, init_g)  # 2019 모든 공에 그룹 안 확률 (그룹 합 1)
P_B = PG_A[:, GIDX] * W_B

def parts(P):  # 24칸 log loss, H/S/C 부분, 그룹 안 부분 (공별)
    y = y_zone[i19]
    pg = (P @ fc.GROUP2)[i9, g_all[i19]]
    return -np.log(np.clip(P[i9, y], 1e-12, None)), -np.log(np.clip(pg, 1e-12, None)), -np.log(np.clip(P[i9, y] / pg, 1e-12, None))

def game_ci(d, B=1000):
    s = pd.DataFrame({"g": gid_all[i19], "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

ll24 = parts(P24)[0]
rows = {}
for nm, P, nt in [("model_13 (보정 없음)", PZ_TRUE[i19], "-"), ("24칸 한 번에 (model_14 구조)", P24, LOC_ROUNDS18),
                  ("(가) H/S/C 3분류 × 그룹 안 그대로", P_A, b3.best_iteration),
                  ("(나) H/S/C 3분류 × 그룹 안 LightGBM", P_B, f"{b3.best_iteration} / " + " · ".join(f"{k} {v}" for k, v in nt_b.items()))]:
    ll, lg_, lw = parts(P)
    d = ll24 - ll
    lo, hi = game_ci(d)
    rows[nm] = {"트리 수": nt, "24칸 ll": ll.mean(), "H/S/C 부분": lg_.mean(), "그룹 안 부분": lw.mean(), "이득 ×1000 (24칸 한 번에 대비)": d.mean() * 1000,
                "95% 하한": lo * 1000, "95% 상한": hi * 1000, "H/S/C 정확도": ((P @ fc.GROUP2).argmax(1) == g_all[i19]).mean()}
display(pd.DataFrame(rows).T)

24칸 한 번에 = model_14 (재현 확인)


,트리 수,24칸 ll,H/S/C 부분,그룹 안 부분,이득 ×1000 (24칸 한 번에 대비),95% 하한,95% 상한,H/S/C 정확도
model_13 (보정 없음),-,2.887245,1.072323,1.814922,-3.702943,-5.571339,-1.923547,0.40645
24칸 한 번에 (model_14 구조),168.0,2.883542,1.070715,1.812827,0.0,0.0,0.0,0.40593
(가) H/S/C 3분류 × 그룹 안 그대로,138.0,2.88519,1.070268,1.814922,-1.648419,-3.395231,0.112336,0.409422
(나) H/S/C 3분류 × 그룹 안 LightGBM,138 / H 43 · S 23 · C 382,2.881426,1.070268,1.811158,2.115956,0.863984,3.411046,0.409422


#### 🔍 0-1 결과: 단계적 분류 (나)가 24칸 한 번에보다 좋음 (구종과 다름)
| 2019 (실제 구종) | 트리 수 | 24칸 ll | H/S/C 부분 | 그룹 안 부분 | 이득 ×1000 (24칸 한 번에 대비, 95%) | H/S/C 정확도 |
| --- | --- | --- | --- | --- | --- | --- |
| model_13 (보정 없음) | – | 2.88725 | 1.07232 | 1.81492 | −3.70 (−5.57 ~ −1.92) | 0.406 |
| 24칸 한 번에 (model_14 구조) | 168 | 2.88354 | 1.07072 | 1.81283 | 0 | 0.406 |
| (가) 3분류 × 그룹 안 그대로 | 138 | 2.88519 | 1.07027 | 1.81492 | −1.65 (−3.40 ~ +0.11) | 0.409 |
| **(나) 3분류 × 그룹 안 LightGBM** | 138 / H 43 · S 23 · C 382 | **2.88143** | 1.07027 | 1.81116 | **+2.12** (+0.86 ~ +3.41) | 0.409 |

- 24칸 한 번에의 상황 이득(model_13 대비 3.7)은 H/S/C 부분 1.6 + 그룹 안 부분 2.1로 나뉨 -> 상황은 존 안/밖뿐 아니라 그룹 안 위치(높낮이·좌우)에도 효과
- (가): H/S/C 부분은 24칸 한 번에보다 조금 좋지만 (1.07027 vs 1.07072) 그룹 안을 안 고쳐서 전체는 나쁨
- **(나)**: H/S/C 부분·그룹 안 부분 모두 24칸 한 번에보다 좋음 -> 학습 데이터가 2018 한 해뿐이라 24칸을 한 번에 배우기보다 나눠 배우는 게 효율적인 것으로 보임 (구종 model_08는 학습 두 해, 9구종)
  - 그룹 안 C(존 밖 8칸) 모델이 트리 382개로 가장 많이 씀 -> 존 밖으로 갈 때 어느 쪽으로 빠지는지가 상황에 따라 달라짐
- 주의: 세 단계 모델의 트리 수는 2019 조기 종료 (model_14 트리 수와 같은 방식), 파라미터는 model_14 값 그대로 (따로 고르지 않음)


## 모델

In [3]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 구종·위치 정규분포는 model_16 값 고정
- 위치 상황 보정: 단계적 분류 (네 모델 파라미터는 3-2에서 고름), 월 단위 재학습

In [4]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 정규분포: model_13 값 (고정)
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 0.707  # 시즌 초 배수
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k)
LOC_VAR = (200.0, 9.74, False, 316.0)  # 분산 따로 (다른 카운트 퍼짐 반감, k_var, unc, 투수 안 분산 당김 k0)
# ② 위치 상황 보정 (model_14): 24칸 LightGBM, 출발점 = model_13 위치 log 확률
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # 상황 변수 (0-3 묶음 4)
LOC_FEAT = CLASSES + ["same_side"] + LOC_SIT  # 구종 원-핫·타자 손 + 상황
LOC_TRAIN_FIRST = 2018  # 위치 보정 학습 첫 해 (2017은 model_13 위치 없음)
LOC_STAGE = ["H/S/C", "H", "S", "C"]  # 단계적 분류 모델 (model_17): H/S/C 3분류 + 그룹 안 3개, 파라미터는 3-2에서 고름
LOC_RETRAIN = "month"  # 상황 보정 재학습 주기 (model_16): 그 달 첫 경기 전날까지 데이터로 다시 학습
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [5]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m13 = pd.read_parquet("results/model_13/per_pitch.parquet")  # 위치 정규분포 재현 확인용
m14 = pd.read_parquet("results/model_14/per_pitch.parquet")  # 24칸 한 번에 시즌 단위 (참고)
m16 = pd.read_parquet("results/model_16/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [6]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [7]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_17    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_17    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: model_13 정규분포 + 상황 보정 (단계적 분류, 월 단위)

### 3-1. model_13 위치 (그대로)

In [9]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g


LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)  # 실제 구종 기준 model_13 24칸 확률 (2018–2020)
for yr in range(LOC_TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    ll_ = -np.log(np.clip(PZ_TRUE[msk][np.arange(msk.sum()), y_zone[msk]], 1e-12, None))
    assert np.allclose(ll_, fc.align(m13, fp, yr)["ll_zone_true"].to_numpy()), f"{yr} model_13 위치 재현 실패"
print("실제 구종 기준 24칸 = model_13 (재현 확인), 2018 계산된 공:", int(np.isfinite(PZ_TRUE[fp["GYEAR"].eq(2018).to_numpy()]).all(1).sum()))

실제 구종 기준 24칸 = model_13 (재현 확인), 2018 계산된 공: 12395


### 3-2. 네 모델 파라미터 고르기 (2019)

- 24칸 log loss = H/S/C 부분 + 그룹 안 부분, 네 모델은 서로 독립 -> 각 모델이 맡은 몫(2019 전체 공 기준 평균)으로 따로 고름
  - H/S/C 3분류: H/S/C 부분 / 그룹 안 g: 실제 그룹이 g인 공의 그룹 안 부분 합 / 전체 공 수
- 학습 2018 -> 2019 조기 종료로 트리 수, 잎 수 · 잎당 최소 공 · lambda_l2 격자 (범위 끝이면 넓힘), 1등과 0.0002 안이면 잎 적게 -> 트리 적게
- 대조: 0장과 같은 model_14 값 (잎 7 · 최소 20 · λ 1000)

In [10]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN -> None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

def game_ci(d, gid, B=1000):  # 경기 단위 부트스트랩 95% 구간 (같은 경기 공은 독립이 아님)
    s = pd.DataFrame({"g": gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])




def auc(p, o):  # Mann-Whitney AUC (같은 값은 평균 순위)
    r = pd.Series(p).rank().to_numpy()
    n1, n0 = o.sum(), len(o) - o.sum()
    return (r[o == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

from scipy.special import log_softmax
for fn in (fc.situation_features, fc.opponent_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(LOC_TRAIN_FIRST)], n=150)
    assert bad.empty, bad
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]]], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)  # 득점권인데 1루 비었음
XLOC = XLOC[LOC_FEAT]
NZ = len(ZONES)
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)



GIDX = pd.Series(fc.ZONE_GROUP).map({"H": 0, "S": 1, "C": 2}).to_numpy()  # 칸 -> 그룹 번호
g_all = GIDX[y_zone]  # 공마다 실제 그룹
ZK = {g: np.flatnonzero(GIDX == gk) for gk, g in enumerate(fc.HSC)}  # 그룹 -> 칸 번호

def center(L):  # 공마다 상수 빼기
    return L - L.mean(1, keepdims=True)

def stage_inputs(P):  # model_13 24칸 확률 P (n, 24) -> 모델별 출발점과 정답 준비: H/S/C 확률·그룹 안 비율
    PG = P @ fc.GROUP2
    W = P / np.clip(PG[:, GIDX], 1e-12, None)
    return PG, W

PG_T, W_T = stage_inputs(PZ_TRUE)  # 실제 구종 기준
INIT = {"H/S/C": center(np.log(np.clip(PG_T, 1e-6, None)))}
YS = {"H/S/C": g_all}
for gk, g in enumerate(fc.HSC):
    INIT[g] = center(np.log(np.clip(W_T[:, ZK[g]], 1e-6, None)))
    yy = np.full(len(fp), -1)
    inz = np.isin(y_zone, ZK[g])
    yy[inz] = pd.Series(np.arange(len(ZK[g])), index=ZK[g]).reindex(y_zone[inz]).to_numpy()  # 그룹 안 칸 번호
    YS[g] = yy
NCLS = {"H/S/C": 3, **{g: len(ZK[g]) for g in fc.HSC}}

def rows_of(stage, mask):  # 그 모델이 학습·채점하는 공 (그룹 안 모델은 실제 그룹이 그 그룹인 공만)
    return np.flatnonzero(mask & ((g_all == fc.HSC.index(stage)) if stage != "H/S/C" else True))

def fit_stage(stage, tr, lp, rounds=None, va=None):
    p = {**BASE_L, "objective": "multiclass", "num_class": NCLS[stage], **lp}
    dtr = lgb.Dataset(XLOC.iloc[tr], YS[stage][tr], init_score=INIT[stage][tr], free_raw_data=False)
    if rounds is None:
        dva = lgb.Dataset(XLOC.iloc[va], YS[stage][va], init_score=INIT[stage][va], reference=dtr)
        return lgb.train(p, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train(p, dtr, rounds)

n19 = int(val.sum())
def score_stage(stage):  # 파라미터 한 조합 -> 그 모델이 맡은 2019 몫 (전체 공 기준), 트리 수
    tr, va = rows_of(stage, yr_l == 2018), rows_of(stage, val)
    def score(num_leaves, min_data_in_leaf, lambda_l2):
        lp = {"num_leaves": int(num_leaves), "min_data_in_leaf": int(min_data_in_leaf), "lambda_l2": float(lambda_l2)}
        bst = fit_stage(stage, tr, lp, va=va)
        raw = bst.predict(XLOC.iloc[va], raw_score=True, num_iteration=bst.best_iteration) + INIT[stage][va]
        nll = -log_softmax(raw, axis=1)[np.arange(len(va)), YS[stage][va]]
        return {"트리 수": bst.best_iteration, "logloss": nll.sum() / n19}
    return score

GRID_S = {"num_leaves": [3, 7, 15], "min_data_in_leaf": [20, 50, 200], "lambda_l2": [300.0, 1000.0, 3000.0]}
STAGE_P, STAGE_N, SEL = {}, {}, {}
for stage in LOC_STAGE:
    sc = score_stage(stage)
    res_s = grid_search(sc, GRID_S)
    tried = {k: list(v) for k, v in GRID_S.items()}
    for rnd in range(3):  # 1등이 범위 끝이면 넓힘 (잎 수 아래쪽 끝 3은 넓히지 않음: 2잎 = 가지 하나)
        b = res_s.iloc[0]
        add = {}
        if b.lambda_l2 == max(tried["lambda_l2"]):
            add["lambda_l2"] = [b.lambda_l2 * 3]
        elif b.lambda_l2 == min(tried["lambda_l2"]):
            add["lambda_l2"] = [b.lambda_l2 / 3]
        if b.num_leaves == max(tried["num_leaves"]):
            add["num_leaves"] = [int(b.num_leaves * 2)]
        if b.min_data_in_leaf == max(tried["min_data_in_leaf"]):
            add["min_data_in_leaf"] = [int(b.min_data_in_leaf * 4)]
        elif b.min_data_in_leaf == min(tried["min_data_in_leaf"]):
            add["min_data_in_leaf"] = [5]
        add = {k: v for k, v in add.items() if v[0] not in tried[k]}
        if not add:
            break
        for k, v in add.items():
            tried[k] += v
            res_s = pd.concat([res_s, grid_search(sc, {**{kk: [b[kk]] for kk in GRID_S}, k: v})])
        res_s = res_s.drop_duplicates(list(GRID_S)).sort_values("logloss").reset_index(drop=True)
    ok_ = res_s[res_s["logloss"] <= res_s["logloss"].iloc[0] + 2e-4]
    pick = ok_.sort_values(["num_leaves", "트리 수"]).iloc[0]  # 평평하면 잎 적게 -> 트리 적게
    STAGE_P[stage] = {"num_leaves": int(pick.num_leaves), "min_data_in_leaf": int(pick.min_data_in_leaf), "lambda_l2": float(pick.lambda_l2)}
    STAGE_N[stage] = int(pick["트리 수"])
    ctl = res_s[(res_s.num_leaves == 7) & (res_s.min_data_in_leaf == 20) & (res_s.lambda_l2 == 1000)]
    SEL[stage] = {"1등": res_s.logloss.iloc[0], "선택": pick.logloss, "대조 (잎 7·20·1000)": ctl.logloss.iloc[0] if len(ctl) else np.nan,
                  **STAGE_P[stage], "트리 수": STAGE_N[stage], "넓힌 값": str({k: v for k, v in tried.items() if v != GRID_S[k]})}
display(pd.DataFrame(SEL).T)
tot = sum(SEL[s]["선택"] for s in LOC_STAGE)
print(f"2019 24칸 log loss (실제 구종, 시즌 단위): 단계적 분류 선택 {tot:.5f} / 대조 {sum(SEL[s]['대조 (잎 7·20·1000)'] for s in LOC_STAGE):.5f} / 24칸 한 번에 (model_14) {fc.align(m14, fp, fc.VAL_YEAR)['ll_zone_true'].mean():.5f}")

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,1등,선택,대조 (잎 7·20·1000),num_leaves,min_data_in_leaf,lambda_l2,트리 수,넓힌 값
H/S/C,1.069921,1.069921,1.070268,3,20,100.0,139,"{'min_data_in_leaf': [20, 50, 200, 5], 'lambda..."
H,0.326259,0.326273,0.326308,3,20,300.0,22,"{'lambda_l2': [300.0, 1000.0, 3000.0, np.float..."
S,0.863128,0.863179,0.863191,3,50,300.0,9,"{'min_data_in_leaf': [20, 50, 200, 800], 'lamb..."
C,0.621474,0.621521,0.621658,3,50,33.333333,79,"{'lambda_l2': [300.0, 1000.0, 3000.0, np.float..."


2019 24칸 log loss (실제 구종, 시즌 단위): 단계적 분류 선택 2.88089 / 대조 2.88143 / 24칸 한 번에 (model_14) 2.88354


#### 선택: 네 모델 모두 잎 3, 규제는 model_14보다 약하게
| 모델 | 잎 · 잎당 최소 · λ | 트리 수 | 맡은 몫 (2019, 시즌 단위) | 대조 (잎 7 · 20 · 1000) |
| --- | --- | --- | --- | --- |
| H/S/C 3분류 | 3 · 20 · 100 | 139 | 1.06992 | 1.07027 |
| 그룹 안 H | 3 · 20 · 300 | 22 | 0.32627 | 0.32631 |
| 그룹 안 S | 3 · 50 · 300 | 9 | 0.86318 | 0.86319 |
| 그룹 안 C | 3 · 50 · 33.3 | 79 | 0.62152 | 0.62166 |

- 2019 24칸 log loss (실제 구종, 시즌 단위): 선택 2.88089 / 대조 2.88143 / 24칸 한 번에 (model_14) 2.88354
- 잎 3이 네 모델 모두 후보 아래쪽 끝 (2잎은 안 봄), C의 λ 33.3도 넓힌 범위의 끝 -> 범위 끝 문제가 남아 있음
- 그룹 안 S는 트리 9개뿐 (거의 보정 안 함)

### 3-3. 월 단위 재학습

- 달마다 그 달 첫 경기 전날까지 2018부터의 모든 초구로 네 모델을 다시 학습 (파라미터·트리 수는 3-2 값), 3월은 4월에·10월은 9월에
- 구종 t를 가정한 보정: 구종 원-핫을 t로 바꾸고, 출발점은 그 구종 가정 model_13 확률에서 계산

In [11]:
DATE_L = fp["game_date"].to_numpy()
MON_L = np.clip(fp["game_date"].dt.month.to_numpy(), 4, 9)
BST_S, plan = {}, []  # (연도, 달, 모델) -> LightGBM
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_l == yr)
    for mon in np.unique(MON_L[iy]):
        start = DATE_L[iy[MON_L[iy] == mon]].min()
        before = (yr_l >= LOC_TRAIN_FIRST) & (DATE_L < start)
        for stage in LOC_STAGE:
            tr = rows_of(stage, before)
            assert DATE_L[tr].max() < start
            BST_S[(yr, mon, stage)] = fit_stage(stage, tr, STAGE_P[stage], STAGE_N[stage])
        plan.append({"연도": yr, "월": mon, "학습 초구": int(before.sum()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))

def loc_correct(P, q, year, t=None):  # model_13 24칸 확률 P (n, 24) -> 단계적 상황 보정 확률 (공마다 그 달 모델), t = 가정한 구종
    X = XLOC.loc[q.index].copy()
    if t is not None:
        X.loc[:, CLASSES] = np.tile((np.array(CLASSES) == t).astype(float), (len(X), 1))
    PG, W = stage_inputs(P)
    out = np.empty_like(P)
    mon = MON_L[fp.index.get_indexer(q.index)]
    for mo in np.unique(mon):
        k = mon == mo
        Xk = X[k]
        pg = np.exp(log_softmax(BST_S[(year, mo, "H/S/C")].predict(Xk, raw_score=True) + center(np.log(np.clip(PG[k], 1e-6, None))), axis=1))
        for gk, g in enumerate(fc.HSC):
            w = np.exp(log_softmax(BST_S[(year, mo, g)].predict(Xk, raw_score=True) + center(np.log(np.clip(W[k][:, ZK[g]], 1e-6, None))), axis=1))
            out[np.ix_(np.flatnonzero(k), ZK[g])] = pg[:, [gk]] * w
    return out

학습 초구       예측 시작
연도   월                   
2019 4  12395  2019-03-23
     5  14963  2019-05-01
     6  17366  2019-06-01
     7  19859  2019-07-02
     8  21650  2019-08-01
     9  24012  2019-09-01
2020 5  25853  2020-05-05
     6  28117  2020-06-02
     7  30496  2020-07-01
     8  32729  2020-08-01
     9  34714  2020-09-01

### 3-4. 진단 (실제 구종 기준)

- model_16 대비 이득: 전체·달별·주자 상황, 24칸 log loss를 H/S/C 부분과 그룹 안 부분으로 나눠서

In [12]:
def runner3(q):
    return np.select([(q["base2"] + q["base3"]) > 0, q["base1"] > 0], ["득점권", "1루만"], "없음")

rows = []
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz = np.arange(len(q)), y_zone[msk]
    P21 = loc_correct(PZ_TRUE[msk], q, yr)
    a20 = fc.align(m16, fp, yr)
    ll20, ll21 = a20["ll_zone_true"].to_numpy(), -np.log(np.clip(P21[i, yz], 1e-12, None))
    pg21 = (P21 @ fc.GROUP2)[i, GIDX[yz]]
    oc = (fc.ZONE_GROUP[yz] == "C").astype(float)
    d = pd.DataFrame({"월": MON_L[msk], "주자": runner3(SIT_L.loc[q.index]), "이득": (ll20 - ll21) * 1000})
    for var in ["월", "주자"]:
        for v, g in d.groupby(var):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "이득 ×1000 (model_16 대비)": g["이득"].mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "이득 ×1000 (model_16 대비)": d["이득"].mean(),
                 "24칸 ll model_17": ll21.mean(), "H/S/C 부분": -np.log(np.clip(pg21, 1e-12, None)).mean(),
                 "그룹 안 부분": (ll21 + np.log(np.clip(pg21, 1e-12, None))).mean(),
                 "존 밖 AUC": auc(P21[:, fc.ZONE_GROUP == "C"].sum(1), oc), "H/S/C 정확도": ((P21 @ fc.GROUP2).argmax(1) == GIDX[yz]).mean()})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(4))

공          이득 ×1000 (model_16 대비)         24칸 ll model_17         H/S/C 부분        그룹 안 부분         존 밖 AUC         H/S/C 정확도        
연도         2019     2020                   2019    2020            2019    2020     2019   2020    2019    2020    2019    2020      2019    2020
변수 값                                                                                                                                             
월  4     2568.0      NaN                 5.0895     NaN             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN
   5     2403.0   2264.0                 4.8624  1.0123             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN
   6     2493.0   2379.0                 1.0673  2.2989             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN
   7     1791.0   2233.0                 1.1089 -0.5419             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN
   8     2362.0   1985.0                -0.8724 -2.6116             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN
   9     1841.0   3754.0                 2.8000 -0.4807             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN
전체      13458.0  12615.0                 2.4146 -0.0347          2.8795  2.8836   1.0697  1.072  1.8098  1.8116  0.5626  0.5548    0.4114  0.4052
주자 1루만   2886.0   2723.0                 1.4798 -0.3205             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN
   득점권   3346.0   3136.0                 3.3985 -2.2776             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN
   없음    7226.0   6756.0                 2.3323  1.1216             NaN     NaN      NaN    NaN     NaN     NaN     NaN     NaN       NaN     NaN

#### 🔍 3-4 결과: 2019는 좋아지지만 2020은 0
- model_16 대비 이득 ×1000 (실제 구종): 2019 +2.41, **2020 −0.03**
- 달별: 2019 +5.1 · +4.9 · +1.1 · +1.1 · −0.9 · +2.8, 2020 +1.0 · +2.3 · −0.5 · −2.6 · −0.5 -> 2020 후반 달은 −
- 주자: 2020 득점권 −2.3, 주자 없음 +1.1
- model_17 2020: H/S/C 부분 1.0720 · 그룹 안 부분 1.8116, 존 밖 AUC 0.555 (model_16 0.553) / 실전 H/S/C 정확도는 0.398 -> 0.397

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_12과 같음, 위치만 3-1에서 고른 값 (분산 따로 포함)

In [13]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
        P = loc_correct(P, q, year, t)  # 상황 보정 (model_14), 정규분포 g는 model_13 그대로
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [14]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종·좌표(정규분포)는 model_16과 같아야 함, 24칸만 바뀜
chk = pp.merge(m16, on=fc.KEYS, suffixes=("", "_16"))
same = ["ll_type", "nll_coord_true", "dist_true"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_16"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_16과 다른 열: {bad}"
print("구종·좌표 = model_16 (재현 확인), 24칸 확률만 바뀜")

(26073, 48)
구종·좌표 = model_16 (재현 확인), 24칸 확률만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_16과 나란히 / 경기 단위 부트스트랩 (model_17 − model_16, 2020)
- 2020은 확인용, 너클 뺀 값도 봄

In [15]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m16_sum = eval_years(m16)
display(pd.concat({MODEL: summary, "model_16": m16_sum, "차이 (model_16 대비)": summary - m16_sum}, axis=1).round(4))

model_17          model_16          차이 (model_16 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9389   2.9402   2.9413   2.9402          -0.0024  0.0000
24칸 top1              0.1323   0.1313   0.1329   0.1331          -0.0005 -0.0018
24칸 top3              0.3023   0.2992   0.2992   0.2999           0.0031 -0.0007
H/S/C logloss         1.0723   1.0741   1.0733   1.0742          -0.0010 -0.0001
H/S/C 정확도             0.4028   0.3973   0.4010   0.3981           0.0019 -0.0008
24칸 logloss (실제 구종)   2.8795   2.8836   2.8819   2.8836          -0.0024  0.0000
구종×칸 logloss          3.9407   3.9397   3.9432   3.9397          -0.0024  0.0000
좌표 평균거리 cm           32.6377  31.8198  32.6377  31.8198           0.0000  0.0000
좌표 NLL                9.3124   9.2662   9.3124   9.2662           0.0000  0.0000

In [16]:
display(fc.bootstrap_diff(fc.pitch_scores(m16, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    a20b, a20 = fc.align(m16, fp, yr), fc.align(pp, fp, yr)
    nk = a20b["pitch_type"].ne("KNUC").to_numpy()
    d_true = a20["ll_zone_true"].to_numpy() - a20b["ll_zone_true"].to_numpy()
    print(f"{yr} 24칸 실제 구종 차이 (model_17 − model_16): 전체 {d_true.mean():+.5f}, 너클 뺌 {d_true[nk].mean():+.5f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0,0.0000,0.0000,차이 불확실
ll_zone,0.0,-0.0013,0.0013,차이 불확실
ll_joint,0.0,-0.0013,0.0014,차이 불확실
nll_coord,0.0,0.0000,0.0000,차이 불확실
dist,0.0,0.0000,0.0000,차이 불확실


2019 24칸 실제 구종 차이 (model_17 − model_16): 전체 -0.00241, 너클 뺌 -0.00241
2020 24칸 실제 구종 차이 (model_17 − model_16): 전체 +0.00003, 너클 뺌 +0.00004


### 🥇1등🥇
#### 2019 24칸 실제 구종 −0.0024였지만 2020은 +0.00003 (0) -> 채택 안 함
- 부트스트랩 (2020, model_17 − model_16): 6개 점수 모두 차이 불확실 -> 형식상 조건은 만족
- 채택 안 한 이유: 2020 효과가 0, 2019 숫자는 네 모델 파라미터를 각각 2019로 고르고 범위를 여러 번 넓힌 결과 (model_15와 같은 종류), 2020 후반 달은 − (−0.5 · −2.6 · −0.5)
- 범위 끝 문제 남음 (잎 3이 네 모델 모두 아래쪽 끝, C의 λ 33.3도 끝) -> 더 넓히면 2019에 더 맞출 뿐으로 봄
- 구종 model_08 (4-17)와 같은 결론: 24칸 한 번에 학습한 모델이 이미 H/S/C 구조를 배움
- **채택 안 함 -> model_16이 계속 비교 출발점** (사용자 결정 2026-11-11)

## 6. 저장

- `results/model_17/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [17]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_17 ['per_pitch.parquet', 'summary.csv']
